# Góp Mầm — Huấn luyện các mô hình AI trên dữ liệu công khai thật (Google Colab)

Notebook này thay dữ liệu **tự sinh** bằng các bộ dữ liệu công khai cùng loại bài toán, rồi xuất đúng các file mà `ai-service` đang đọc:

| Mô hình | Dữ liệu | File xuất ra (`models/`) |
|---|---|---|
| Dự đoán khả năng thành công | Kickstarter Projects (Kaggle, ~378k dự án 2009–2018) | `success_model.pkl`, `success_features.json`, `success_metrics.json`, `success_model.metadata.json` |
| Phát hiện giao dịch bất thường | IEEE-CIS Fraud Detection (Kaggle, ~590k giao dịch thật, ~3,5% gian lận) | `fraud_classifier.pkl`, `fraud_score_reference.json`, `fraud_metrics.json` |
| Gợi ý dự án (độ tương đồng lĩnh vực) | DonorsChoose.org (Kaggle `hanselhansel/donorschoose`) | `category_similarity.json`, `recommender_metrics.json` |
| Dự đoán riêng cho **Giáo dục** (thử nghiệm, chưa triển khai) | DonorsChoose.org — dự án đủ tiền / hết hạn | `education_success_model.pkl`, `education_success_features.json`, `education_success_metrics.json`, `education_success_model.metadata.json`, `education_diagnostics.png` |

**Cách chạy:** *Runtime → Run all*. Lần đầu notebook cài thư viện rồi **tự khởi động lại phiên** (Colab báo phiên bị dừng là bình thường) — khi đó bấm **Run all** thêm lần nữa; lần hai bỏ qua bước cài và chạy tới hết. Toàn bộ mất khoảng 20–40 phút; cuối cùng tự tải `gopmam_models.zip`.

**Chuẩn bị (một lần):**

1. Token Kaggle: điền vào ô `KAGGLE_API_TOKEN` ở mục 1, **hoặc** lưu trong *Colab → biểu tượng chìa khóa (Secrets)* với tên `KAGGLE_API_TOKEN` và bật *Notebook access*.
2. Chấp nhận luật cuộc thi: <https://www.kaggle.com/competitions/ieee-fraud-detection/rules> (bắt buộc để tải IEEE-CIS).

**Nguyên tắc để kết quả đáng tin cậy (dùng khi bảo vệ)**

- Chỉ dùng đặc trưng **có tại thời điểm tạo chiến dịch / phát sinh giao dịch**; không dùng số tiền đã gây được, số người ủng hộ cuối kỳ… (tránh rò rỉ nhãn).
- Chia train/test **theo thời gian** (80% cũ → train, 20% mới → test), giống cách hệ thống dự đoán tương lai.
- Luôn so với **mốc đơn giản** (đoán lớp đa số, ngẫu nhiên, phổ biến) và báo cáo nhiều chỉ số, không chỉ Accuracy.
- Ghi rõ **độ lệch miền**: dữ liệu Mỹ/quốc tế, nền tảng chạy ở Việt Nam → kết quả AI chỉ để tham khảo, con người quyết định.

## 0. Cài đúng phiên bản thư viện

`ai-service` ghim phiên bản trong `requirements.txt`; file `.pkl` tạo bằng phiên bản khác có thể **không nạp được** ở server. Cell dưới chỉ cài khi phiên bản chưa đúng. Cảnh báo *"pip's dependency resolver…"* về `google-colab`/`numba` là bình thường, không ảnh hưởng.

In [ ]:
import importlib.metadata as metadata
import os
import subprocess
import sys

PINNED = {"scikit-learn": "1.9.1", "numpy": "2.5.4", "pandas": "3.0.6", "joblib": "1.6.0"}


def installed_version(package: str) -> str | None:
    try:
        return metadata.version(package)
    except metadata.PackageNotFoundError:
        return None


missing = [
    f"{name}=={version}" for name, version in PINNED.items() if installed_version(name) != version
]
if missing:
    print("Cài:", missing)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)
# kaggle >= 2.2 mới nhận token dạng KGAT_...; matplotlib để vẽ biểu đồ
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "kaggle"], check=True)
if installed_version("matplotlib") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "matplotlib"], check=True)
if missing:
    # Colab đã nạp sẵn bản cũ vào bộ nhớ: phải khởi động lại trước khi import, nếu không sẽ lỗi
    # kiểu "cannot import name ... from numpy". Lần Run all thứ hai sẽ bỏ qua bước cài này.
    print(
        "\n>>> Đã cài xong. Phiên sẽ tự khởi động lại — sau đó bấm Runtime → Run all lần nữa. <<<"
    )
    os.kill(os.getpid(), 9)
print("Thư viện đã đúng phiên bản")

### 0.1. Kiểm tra phiên bản và cấu hình chung

In [ ]:
import json
import os
import shutil
import subprocess
import zipfile
from datetime import UTC, datetime
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn

EXPECTED = {"sklearn": "1.9.1", "numpy": "2.5.4", "pandas": "3.0.6", "joblib": "1.6.0"}
ACTUAL = {
    "sklearn": sklearn.__version__,
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "joblib": joblib.__version__,
}
for name, version in EXPECTED.items():
    status = "OK" if ACTUAL[name] == version else "SAI PHIÊN BẢN"
    print(f"{name:8s} {ACTUAL[name]:10s} (cần {version}) {status}")
if ACTUAL != EXPECTED:
    # Colab đã nạp phiên bản cũ trước khi cài: khởi động lại phiên để dùng bản mới.
    print("\n>>> Phiên sẽ tự khởi động lại. Sau đó bấm Runtime → Run all thêm một lần nữa. <<<")
    os.kill(os.getpid(), 9)

SEED = 42
TRAIN_FRACTION = 0.8
MODEL_VERSION = "2026.11-real.2"
# Tỷ giá quy đổi mục tiêu USD -> VNĐ (cố định để tái lập; ghi vào metrics).
USD_TO_VND = 25_000

# Danh mục lĩnh vực của ai-service (app/services/catalog.py) — thứ tự = category_id.
CATEGORIES = ["Môi trường", "Khởi nghiệp", "Giáo dục", "Y tế", "Văn hóa", "Công nghệ"]
CATEGORY_IDS = {name: idx for idx, name in enumerate(CATEGORIES)}

# 5 đặc trưng mô hình gian lận (FRAUD_FEATURES trong catalog.py) — PHẢI giữ đúng thứ tự.
FRAUD_FEATURES = [
    "contribution_count_1h",
    "failed_payment_count",
    "total_payment_count",
    "amount_z_score",
    "new_account_days",
]

DATA_DIR = Path("/content/data")
OUT_DIR = Path("/content/models")
DATA_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)


def now_iso() -> str:
    return datetime.now(UTC).replace(microsecond=0).isoformat()


def save_json(name: str, payload) -> None:
    (OUT_DIR / name).write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")
    print("Đã ghi", OUT_DIR / name)

## 1. Kết nối Kaggle và tải dữ liệu

Token được lấy theo thứ tự: ô `KAGGLE_API_TOKEN` bên dưới → Colab Secrets (`KAGGLE_API_TOKEN`) → hỏi khi chạy.

> ⚠️ Không chia sẻ hoặc đưa notebook đã điền token lên GitHub. Sau khi huấn luyện xong nên vào Kaggle → Settings → API Tokens để thu hồi token và tạo token mới.

In [ ]:
from getpass import getpass

from google.colab import files

KAGGLE_API_TOKEN = ""  # dán token KGAT_... vào đây, hoặc dùng Colab Secrets


def resolve_kaggle_token() -> str:
    if KAGGLE_API_TOKEN.strip():
        return KAGGLE_API_TOKEN.strip()
    try:
        from google.colab import userdata

        secret = userdata.get("KAGGLE_API_TOKEN")
        if secret:
            return secret.strip()
    except Exception:
        pass
    if os.environ.get("KAGGLE_API_TOKEN"):
        return os.environ["KAGGLE_API_TOKEN"].strip()
    return getpass("Dán Kaggle API token: ").strip()


token = resolve_kaggle_token()
os.environ["KAGGLE_API_TOKEN"] = token
kaggle_dir = Path.home() / ".kaggle"
kaggle_dir.mkdir(exist_ok=True)
(kaggle_dir / "access_token").write_text(token)
os.chmod(kaggle_dir / "access_token", 0o600)

check = subprocess.run(
    ["kaggle", "datasets", "list", "-s", "kickstarter", "--page-size", "1"],
    capture_output=True,
    text=True,
)
if check.returncode != 0:
    raise RuntimeError(f"Xác thực Kaggle thất bại:\n{check.stdout}\n{check.stderr}")
print("Kết nối Kaggle OK")

In [ ]:
SOURCES = {
    # tên thư mục: (loại, slug Kaggle, trang web để tải tay)
    "kickstarter": (
        "dataset",
        "kemical/kickstarter-projects",
        "https://www.kaggle.com/datasets/kemical/kickstarter-projects",
    ),
    "ieee": (
        "competition",
        "ieee-fraud-detection",
        "https://www.kaggle.com/competitions/ieee-fraud-detection/data",
    ),
    "donorschoose": (
        "dataset",
        "hanselhansel/donorschoose",
        "https://www.kaggle.com/datasets/hanselhansel/donorschoose",
    ),
}


def download(name: str) -> Path:
    kind, slug, page = SOURCES[name]
    target = DATA_DIR / name
    target.mkdir(parents=True, exist_ok=True)
    if any(target.glob("*.csv")):
        print(f"[{name}] đã có dữ liệu, bỏ qua tải")
        return target
    command = ["kaggle", "competitions" if kind == "competition" else "datasets", "download"]
    command += ["-c", slug] if kind == "competition" else ["-d", slug]
    command += ["-p", str(target)]
    result = subprocess.run(command, capture_output=True, text=True)
    print(result.stdout[-500:], result.stderr[-500:])
    for archive in target.glob("*.zip"):
        with zipfile.ZipFile(archive) as zf:
            zf.extractall(target)
        archive.unlink()
    if not any(target.rglob("*.csv")):
        raise RuntimeError(
            f"[{name}] chưa có file CSV. Kiểm tra quyền truy cập tại {page} "
            f"(cuộc thi phải bấm Accept luật) hoặc tải tay và giải nén vào {target}"
        )
    return target


for name in SOURCES:
    download(name)
!du -sh /content/data/*

## 2. Mô hình dự đoán khả năng thành công (Kickstarter)

**Nhãn:** `state` = `successful` (1) / `failed` (0); bỏ các trạng thái khác (canceled, live, suspended…).

**Đặc trưng triển khai** — chỉ những gì nền tảng Góp Mầm tính được *ngay khi tạo chiến dịch* và có trong hợp đồng dữ liệu của `ai-service`:

| Đặc trưng | Cách tính từ Kickstarter | Ở Góp Mầm |
|---|---|---|
| `category_id` | `main_category` ánh xạ về 6 lĩnh vực | lĩnh vực chiến dịch |
| `goal_amount` | `usd_goal_real` × tỷ giá → VNĐ | mục tiêu gây quỹ |
| `duration_days` | `deadline` − `launched` (ngày) | thời gian kêu gọi |
| `name_words` | số từ trong `name` | số từ trong tên chiến dịch |
| `launch_month` | tháng của `launched` (1–12) | tháng bắt đầu gây quỹ (giờ Việt Nam) |
| `launch_weekday` | thứ của `launched` (0 = thứ Hai … 6 = Chủ nhật) | thứ bắt đầu gây quỹ |

**Không dùng** `pledged`, `backers`, `usd_pledged*` (kết quả cuối kỳ → rò rỉ nhãn).

Từ `2026.11-real.2` thêm 3 đặc trưng `name_words`, `launch_month`, `launch_weekday`: có sẵn trong Kickstarter và nền tảng tính được ngay khi tạo chiến dịch (tên + ngày bắt đầu), không rò rỉ nhãn. Kết quả của bộ 3 đặc trưng cũ (`2026.10-real.1`) được huấn luyện lại cùng thuật toán, cùng cách chia và ghi vào `success_metrics.json` (`previous_feature_set`) để so sánh.

Thí nghiệm "mở rộng" (thêm quốc gia `is_us` và 15 nhóm gốc thay cho 6 lĩnh vực) **chỉ để báo cáo**, không xuất ra: Góp Mầm luôn ở Việt Nam (`is_us` luôn bằng 0) và chỉ có 6 nhóm lĩnh vực tương ứng.

In [ ]:
# Kickstarter không có lĩnh vực Môi trường / Y tế -> hai lĩnh vực này không có mẫu
# (ghi nhận là hạn chế).
KICKSTARTER_TO_CATEGORY = {
    "Technology": "Công nghệ",
    "Games": "Công nghệ",
    "Design": "Công nghệ",
    "Food": "Khởi nghiệp",
    "Fashion": "Khởi nghiệp",
    "Crafts": "Khởi nghiệp",
    "Publishing": "Giáo dục",
    "Journalism": "Giáo dục",
    "Art": "Văn hóa",
    "Comics": "Văn hóa",
    "Dance": "Văn hóa",
    "Film & Video": "Văn hóa",
    "Music": "Văn hóa",
    "Photography": "Văn hóa",
    "Theater": "Văn hóa",
}
# Thứ tự = thứ tự cột của mô hình, ghi vào success_features.json; ai-service tra theo tên.
SUCCESS_FEATURES = [
    "category_id",
    "goal_amount",
    "duration_days",
    "name_words",
    "launch_month",
    "launch_weekday",
]
# Bộ đặc trưng của mô hình trước (2026.10-real.1) — huấn luyện lại để so sánh.
PREVIOUS_FEATURES = ["category_id", "goal_amount", "duration_days"]


def prepare_kickstarter(raw: pd.DataFrame) -> pd.DataFrame:
    df = raw.copy()
    df["state"] = df["state"].astype(str).str.strip().str.lower()
    df = df[df["state"].isin(["successful", "failed"])]
    df["launched"] = pd.to_datetime(df["launched"], errors="coerce")
    df["deadline"] = pd.to_datetime(df["deadline"], errors="coerce")
    df = df.dropna(subset=["launched", "deadline", "usd_goal_real", "main_category"])
    df = df[df["launched"].dt.year >= 2009]  # loại ngày lỗi 1970
    df = df[df["usd_goal_real"] > 0]
    df["category"] = df["main_category"].map(KICKSTARTER_TO_CATEGORY)
    df = df.dropna(subset=["category"])
    df["category_id"] = df["category"].map(CATEGORY_IDS).astype(int)
    df["goal_amount"] = df["usd_goal_real"].astype(float) * USD_TO_VND
    df["duration_days"] = (
        ((df["deadline"] - df["launched"]).dt.total_seconds() / 86_400).round().clip(1, 92)
    )
    df["success"] = (df["state"] == "successful").astype(int)
    # Đặc trưng triển khai từ 2026.11 — backend tính y hệt (countWords, launchCalendar trong
    # backend/src/modules/ai/ai.service.ts): số từ của tên, tháng 1–12, thứ 0 = thứ Hai.
    df["name_words"] = df["name"].fillna("").astype(str).str.split().str.len()
    df["launch_month"] = df["launched"].dt.month
    df["launch_weekday"] = df["launched"].dt.weekday
    # Chỉ cho thí nghiệm mở rộng (Góp Mầm luôn ở Việt Nam nên không triển khai).
    df["is_us"] = (df["country"].astype(str) == "US").astype(int)
    return df.sort_values("launched").reset_index(drop=True)


def time_split(
    df: pd.DataFrame, fraction: float = TRAIN_FRACTION
) -> tuple[pd.DataFrame, pd.DataFrame]:
    cut = int(len(df) * fraction)
    return df.iloc[:cut], df.iloc[cut:]

In [ ]:
ks_dir = DATA_DIR / "kickstarter"
ks_file = sorted(ks_dir.rglob("ks-projects-2018*.csv")) or sorted(ks_dir.rglob("*.csv"))
print("Đọc", ks_file[-1])
ks_raw = pd.read_csv(
    ks_file[-1],
    usecols=[
        "ID",
        "name",
        "main_category",
        "deadline",
        "launched",
        "state",
        "country",
        "usd_goal_real",
    ],
    encoding_errors="replace",
    low_memory=False,
)
ks = prepare_kickstarter(ks_raw)
ks_train, ks_test = time_split(ks)
print(
    f"Dòng dùng được: {len(ks):,} | train {len(ks_train):,} "
    f"({ks_train['launched'].min():%Y-%m} → {ks_train['launched'].max():%Y-%m})"
    f" | test {len(ks_test):,} (→ {ks_test['launched'].max():%Y-%m})"
)
print(
    "Tỷ lệ thành công train/test:",
    round(ks_train["success"].mean(), 3),
    "/",
    round(ks_test["success"].mean(), 3),
)
print(ks.groupby("category")["success"].agg(["count", "mean"]).round(3))

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import (
    GradientBoostingClassifier,
    HistGradientBoostingClassifier,
    RandomForestClassifier,
)
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    brier_score_loss,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import TimeSeriesSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler


def classification_report_dict(y_true, proba, threshold: float = 0.5) -> dict:
    y_pred = (proba >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    return {
        "roc_auc": float(roc_auc_score(y_true, proba)),
        "pr_auc": float(average_precision_score(y_true, proba)),
        "f1": float(f1_score(y_true, y_pred, zero_division=0)),
        "precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "recall": float(recall_score(y_true, y_pred, zero_division=0)),
        "brier": float(brier_score_loss(y_true, proba)),
        "accuracy": float((y_pred == y_true).mean()),
        "confusion_matrix": {"tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp)},
    }


def log_goal_pipeline() -> Pipeline:
    # Mốc tuyến tính (chỉ để so sánh, không xuất): log(mục tiêu, số ngày), số từ của tên,
    # lĩnh vực/tháng/thứ dạng one-hot. Chỉ số cột theo SUCCESS_FEATURES.
    pre = ColumnTransformer(
        [
            (
                "num",
                Pipeline([("log", FunctionTransformer(np.log1p)), ("scale", StandardScaler())]),
                [1, 2],
            ),
            ("words", StandardScaler(), [3]),
            ("cat", OneHotEncoder(handle_unknown="ignore"), [0, 4, 5]),
        ]
    )
    return Pipeline([("pre", pre), ("lr", LogisticRegression(max_iter=1000))])


# Mô hình triển khai: cây quyết định không cần chuẩn hóa (server gửi giá trị thô)
# và có feature_importances_
# để ai-service giải thích yếu tố ảnh hưởng.
DEPLOYABLE = {
    "random_forest": lambda: RandomForestClassifier(
        # Giới hạn độ sâu để file .pkl gọn (vài chục MB) mà vẫn đủ khớp dữ liệu
        n_estimators=100,
        max_depth=10,
        min_samples_leaf=50,
        n_jobs=-1,
        random_state=SEED,
    ),
    "gradient_boosting": lambda: GradientBoostingClassifier(
        n_estimators=200, max_depth=3, learning_rate=0.1, subsample=0.8, random_state=SEED
    ),
}
BASELINES = {
    "majority_class": lambda: DummyClassifier(strategy="prior"),
    "logistic_regression": log_goal_pipeline,
}

In [ ]:
X_train = ks_train[SUCCESS_FEATURES].to_numpy(dtype=float)
X_test = ks_test[SUCCESS_FEATURES].to_numpy(dtype=float)
y_train = ks_train["success"].to_numpy()
y_test = ks_test["success"].to_numpy()

success_eval, fitted = {}, {}
for name, factory in {**BASELINES, **DEPLOYABLE}.items():
    model = factory().fit(X_train, y_train)
    fitted[name] = model
    success_eval[name] = classification_report_dict(y_test, model.predict_proba(X_test)[:, 1])
    print(
        f"{name:20s} ROC-AUC={success_eval[name]['roc_auc']:.4f} "
        f"PR-AUC={success_eval[name]['pr_auc']:.4f} "
        f"F1={success_eval[name]['f1']:.4f} Brier={success_eval[name]['brier']:.4f}"
    )

best_name = max(DEPLOYABLE, key=lambda n: success_eval[n]["roc_auc"])
best_model = fitted[best_name]
print("\nChọn triển khai:", best_name)

# Bộ 3 đặc trưng cũ: cùng thuật toán, cùng cách chia -> thấy rõ phần đóng góp của 3 đặc trưng mới.
prev_idx = [SUCCESS_FEATURES.index(name) for name in PREVIOUS_FEATURES]
prev_model = DEPLOYABLE[best_name]().fit(X_train[:, prev_idx], y_train)
previous_eval = classification_report_dict(
    y_test, prev_model.predict_proba(X_test[:, prev_idx])[:, 1]
)
gain = success_eval[best_name]["roc_auc"] - previous_eval["roc_auc"]
print(
    f"Bộ 3 đặc trưng cũ: ROC-AUC={previous_eval['roc_auc']:.4f} -> "
    f"bộ {len(SUCCESS_FEATURES)} đặc trưng: {success_eval[best_name]['roc_auc']:.4f} "
    f"({gain:+.4f})"
)
if gain <= 0:
    print(
        ">>> Đặc trưng mới KHÔNG cải thiện ROC-AUC trên tập test. Cân nhắc giữ mô hình cũ "
        "(2026.10-real.1) thay vì dùng bộ file mới. <<<"
    )


In [ ]:
# Kiểm định chéo theo thời gian (5 lần) trên tập train để có khoảng dao động của ROC-AUC.
cv_scores = []
for fold, (tr, va) in enumerate(TimeSeriesSplit(n_splits=5).split(X_train), start=1):
    m = DEPLOYABLE[best_name]().fit(X_train[tr], y_train[tr])
    cv_scores.append(roc_auc_score(y_train[va], m.predict_proba(X_train[va])[:, 1]))
    print(f"Fold {fold}: ROC-AUC={cv_scores[-1]:.4f}")
print(f"ROC-AUC trung bình {np.mean(cv_scores):.4f} ± {np.std(cv_scores):.4f}")

# Thí nghiệm mở rộng: nhiều đặc trưng hơn (KHÔNG xuất) — cho thấy dư địa nếu nền tảng thu thập thêm.
EXTENDED = ["goal_amount", "duration_days", "name_words", "launch_month", "launch_weekday", "is_us"]
ext_train = pd.concat(
    [ks_train[EXTENDED], pd.get_dummies(ks_train["main_category"], dtype=float)], axis=1
)
ext_test = pd.concat(
    [ks_test[EXTENDED], pd.get_dummies(ks_test["main_category"], dtype=float)], axis=1
)
ext_test = ext_test.reindex(columns=ext_train.columns, fill_value=0.0)
ext_model = HistGradientBoostingClassifier(random_state=SEED).fit(ext_train, y_train)
extended_eval = classification_report_dict(y_test, ext_model.predict_proba(ext_test)[:, 1])
print(
    f"\nMở rộng ({ext_train.shape[1]} đặc trưng): ROC-AUC={extended_eval['roc_auc']:.4f} "
    f"PR-AUC={extended_eval['pr_auc']:.4f}"
)

In [ ]:
from sklearn.calibration import calibration_curve

proba_best = best_model.predict_proba(X_test)[:, 1]
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
frac_pos, mean_pred = calibration_curve(y_test, proba_best, n_bins=10)
axes[0].plot(mean_pred, frac_pos, "o-", label=best_name)
axes[0].plot([0, 1], [0, 1], "--", color="gray", label="Hiệu chuẩn hoàn hảo")
axes[0].set(
    title="Đường hiệu chuẩn xác suất (tập test)",
    xlabel="Xác suất dự đoán",
    ylabel="Tỷ lệ thành công thực tế",
)
axes[0].legend()
axes[1].barh(SUCCESS_FEATURES, best_model.feature_importances_)
axes[1].set(title="Mức quan trọng đặc trưng", xlabel="feature_importances_")
plt.tight_layout()
plt.savefig(OUT_DIR / "success_diagnostics.png", dpi=150)
plt.show()

In [ ]:
# Hướng tác động (tương quan với nhãn) — ai-service dùng để ghi "tăng/giảm khả năng thành công".
feature_impact = {}
for idx, name in enumerate(SUCCESS_FEATURES):
    col = X_train[:, idx]
    corr = 0.0 if np.std(col) == 0 else float(np.corrcoef(col, y_train)[0, 1])
    feature_impact[name] = {"corr": round(corr, 6)}

success_metrics = {
    "model_version": MODEL_VERSION,
    "best_model": best_name,
    "chosen_by": "roc_auc",
    "trained_at": now_iso(),
    "dataset": {
        "name": "Kickstarter Projects (Kaggle kemical/kickstarter-projects, "
        "ks-projects-201801.csv)",
        "rows": int(len(ks)),
        "period": f"{ks['launched'].min():%Y-%m-%d} → {ks['launched'].max():%Y-%m-%d}",
        "success_rate": round(float(ks["success"].mean()), 6),
        "usd_to_vnd": USD_TO_VND,
        "category_mapping": KICKSTARTER_TO_CATEGORY,
    },
    "features": SUCCESS_FEATURES,
    "split": {
        "strategy": "time_order",
        "train_rows": int(len(ks_train)),
        "test_rows": int(len(ks_test)),
        "threshold": TRAIN_FRACTION,
        "semantics": "80% dự án ra mắt sớm hơn để huấn luyện, 20% ra mắt sau để kiểm tra "
        "(mô phỏng dự đoán dự án tương lai).",
    },
    "models": success_eval,
    "chosen": success_eval[best_name],
    "time_series_cv_roc_auc": {
        "mean": float(np.mean(cv_scores)),
        "std": float(np.std(cv_scores)),
        "folds": [float(s) for s in cv_scores],
    },
    "previous_feature_set": {
        "model_version": "2026.10-real.1",
        "features": PREVIOUS_FEATURES,
        "metrics": previous_eval,
        "roc_auc_gain": float(gain),
    },
    "extended_experiment": {
        "features": list(ext_train.columns),
        "metrics": extended_eval,
        "note": "Chỉ để báo cáo; không triển khai vì Góp Mầm luôn ở Việt Nam (is_us = 0) "
        "và chỉ có 6 nhóm lĩnh vực.",
    },
    "feature_impact": feature_impact,
    "limitations": [
        "Dữ liệu Kickstarter (chủ yếu Mỹ, 2009–2018), khác thị trường và loại hình "
        "gây quỹ xã hội ở Việt Nam.",
        "Không có mẫu cho lĩnh vực Môi trường và Y tế.",
        "Mục tiêu quy đổi theo tỷ giá cố định, chưa điều chỉnh sức mua.",
        "Tháng/thứ bắt đầu học theo mùa vụ và thói quen ủng hộ ở Mỹ, có thể khác Việt Nam.",
    ],
}

joblib.dump(best_model, OUT_DIR / "success_model.pkl", compress=3)
save_json("success_features.json", SUCCESS_FEATURES)
save_json("success_metrics.json", success_metrics)
save_json(
    "success_model.metadata.json",
    {
        "model_version": MODEL_VERSION,
        "feature_schema_version": "2.1.0",
        "trained_at": success_metrics["trained_at"],
        "features": SUCCESS_FEATURES,
        "best_model": best_name,
    },
)

# Kiểm tra giống cách ai-service gọi: vector giá trị thô theo đúng thứ tự success_features.json
sample = {
    "category_id": CATEGORY_IDS["Giáo dục"],
    "goal_amount": 50_000_000,
    "duration_days": 45,
    "name_words": 7,
    "launch_month": 11,
    "launch_weekday": 0,
}
loaded = joblib.load(OUT_DIR / "success_model.pkl")
print(
    "Xác suất thành công (Giáo dục, 50 triệu, 45 ngày, tên 7 từ, bắt đầu thứ Hai tháng 11):",
    round(
        float(
            loaded.predict_proba(np.asarray([[sample[f] for f in SUCCESS_FEATURES]], dtype=float))[
                0
            ][1]
        ),
        4,
    ),
)

## 3. Mô hình phát hiện giao dịch bất thường (IEEE-CIS Fraud Detection)

Ánh xạ sang 5 đặc trưng của ai-service (thực thể = "khách hàng" ghép từ `card1`, `card2`, `addr1` và ngày mở thẻ suy ra từ `D1`):

| Đặc trưng | Cách tính | Ghi chú |
|---|---|---|
| `contribution_count_1h` | số giao dịch của khách trong 1 giờ gần nhất (gồm giao dịch hiện tại) | tốc độ giao dịch |
| `failed_payment_count` | **0** | IEEE-CIS không có giao dịch thất bại; tín hiệu này do tầng luật xử lý |
| `total_payment_count` | số giao dịch tích lũy của khách tới thời điểm đó | |
| `amount_z_score` | độ lệch số tiền so với lịch sử *trước đó* của khách (\|z\| ≤ 10) | chỉ dùng quá khứ, không rò rỉ |
| `new_account_days` | `D1` (số ngày từ lần đầu dùng thẻ) | thiếu → trung vị |

**Quy trình chọn phương pháp (dùng khi bảo vệ):**

1. Thử **Isolation Forest không giám sát** (phương pháp ban đầu) — nhãn `isFraud` chỉ dùng để chấm. Trên dữ liệu thật kết quả xấp xỉ ngẫu nhiên: "bất thường về thống kê" (vd. khách quen giao dịch nhiều) ≠ "gian lận".
2. Chuyển sang **mô hình có giám sát** (`HistGradientBoostingClassifier`) học từ giao dịch gian lận đã gán nhãn thật, trên cùng 5 đặc trưng.
3. Gian lận chỉ ~3,5% nên xác suất thô luôn nhỏ → ai-service đổi xác suất thành **phân vị rủi ro** ("rủi ro hơn X% giao dịch") theo phân phối điểm lúc huấn luyện (`fraud_score_reference.json`).
4. Mô hình chỉ đóng góp **25%** điểm rủi ro; một mình nó không đẩy được cảnh báo lên mức trung bình — luật nghiệp vụ và quản trị viên quyết định.

In [ ]:
def build_fraud_features(tx: pd.DataFrame) -> pd.DataFrame:
    df = tx.copy()
    day = np.floor(df["TransactionDT"] / 86_400)

    def part(values: pd.Series) -> pd.Series:
        # pandas 3 giữ giá trị thiếu là NaN khi đổi sang chuỗi -> thay bằng "na"
        # để không gộp nhầm các khách hàng khác nhau
        return values.astype("string").fillna("na")

    df["uid"] = (
        part(df["card1"])
        + "_"
        + part(df["card2"])
        + "_"
        + part(df["addr1"])
        + "_"
        + part(day - df["D1"])
    )
    df = df.sort_values(["uid", "TransactionDT"], kind="mergesort").reset_index(drop=True)

    codes = pd.factorize(df["uid"])[0].astype(np.int64)
    t = df["TransactionDT"].to_numpy(dtype=np.int64)
    key = codes * 10**9 + t  # TransactionDT < 1e9 giây -> khóa không chồng lấn giữa các khách
    position = np.arange(len(df))
    first_in_window = np.searchsorted(key, key - 3600, side="right")
    df["contribution_count_1h"] = (position - first_in_window + 1).astype(float)

    df["total_payment_count"] = (df.groupby("uid").cumcount() + 1).astype(float)

    amount = df["TransactionAmt"].astype(float)
    group = df.groupby("uid")["TransactionAmt"]
    prev_n = df["total_payment_count"] - 1
    prev_sum = group.cumsum() - amount
    prev_sq = df.assign(sq=amount**2).groupby("uid")["sq"].cumsum() - amount**2
    prev_mean = prev_sum / prev_n.replace(0, np.nan)
    prev_var = (prev_sq / prev_n.replace(0, np.nan)) - prev_mean**2
    prev_std = np.sqrt(prev_var.clip(lower=0))
    personal_z = (amount - prev_mean) / prev_std.replace(0, np.nan)
    # Chưa đủ lịch sử (< 2 giao dịch trước) -> so với phân phối chung của log(số tiền)
    log_amount = np.log1p(amount)
    global_z = (log_amount - log_amount.mean()) / log_amount.std()
    use_personal = (prev_n >= 2) & personal_z.notna()
    df["amount_z_score"] = np.where(use_personal, personal_z, global_z).clip(-10, 10)

    df["new_account_days"] = df["D1"].astype(float).clip(lower=0)
    df["failed_payment_count"] = 0.0
    return df.sort_values("TransactionDT", kind="mergesort").reset_index(drop=True)

In [ ]:
ieee_dir = DATA_DIR / "ieee"
tx_raw = pd.read_csv(
    next(ieee_dir.rglob("train_transaction.csv")),
    usecols=[
        "TransactionID",
        "isFraud",
        "TransactionDT",
        "TransactionAmt",
        "card1",
        "card2",
        "addr1",
        "D1",
    ],
)
fraud_df = build_fraud_features(tx_raw)
fraud_df["new_account_days"] = fraud_df["new_account_days"].fillna(
    fraud_df["new_account_days"].median()
)
fraud_train, fraud_test = time_split(fraud_df)
fraud_rate = float(fraud_train["isFraud"].mean())
print(
    f"Giao dịch: {len(fraud_df):,} | train {len(fraud_train):,} | test {len(fraud_test):,} "
    f"| tỷ lệ gian lận train {fraud_rate:.4f}"
)
print(fraud_df[FRAUD_FEATURES + ["isFraud"]].groupby("isFraud").mean().round(3).T)

In [ ]:
from sklearn.ensemble import IsolationForest

Xf_train = fraud_train[FRAUD_FEATURES].to_numpy(dtype=float)
Xf_test = fraud_test[FRAUD_FEATURES].to_numpy(dtype=float)
yf_train = fraud_train["isFraud"].to_numpy()
yf_test = fraud_test["isFraud"].to_numpy()
is_fraud = yf_test == 1
fraud_rate_test = float(is_fraud.mean())


def top_k_report(score: np.ndarray) -> dict:
    # Trong k% giao dịch bị chấm rủi ro cao nhất có bao nhiêu % là gian lận thật
    # (và gấp mấy lần chọn ngẫu nhiên)
    report = {}
    for share in (0.01, 0.05):
        k = max(1, int(len(score) * share))
        top = np.argsort(-score)[:k]
        precision = float(is_fraud[top].mean())
        report[f"top_{int(share * 100)}pct"] = {
            "precision": precision,
            "lift": precision / fraud_rate_test,
            "recall": float(is_fraud[top].sum() / max(1, is_fraud.sum())),
        }
    return report


def score_report(score: np.ndarray) -> dict:
    return {
        "roc_auc": float(roc_auc_score(yf_test, score)),
        "pr_auc": float(average_precision_score(yf_test, score)),
        "pr_auc_random_baseline": fraud_rate_test,
        **top_k_report(score),
    }


def make_classifier() -> HistGradientBoostingClassifier:
    return HistGradientBoostingClassifier(
        max_iter=300, learning_rate=0.05, early_stopping=True, random_state=SEED
    )


# (1) Isolation Forest không giám sát — đã thử, giữ kết quả để báo cáo
scaler = StandardScaler().fit(Xf_train)
forest = IsolationForest(n_estimators=200, contamination=fraud_rate, random_state=SEED, n_jobs=-1)
forest.fit(scaler.transform(Xf_train))
iso_report = score_report(-forest.decision_function(scaler.transform(Xf_test)))
print("Isolation Forest (đã thử, loại):", json.dumps(iso_report, indent=2))

# (2) Mô hình có giám sát — triển khai
classifier = make_classifier().fit(Xf_train, yf_train)
clf_report = score_report(classifier.predict_proba(Xf_test)[:, 1])
print("Có giám sát (triển khai):", json.dumps(clf_report, indent=2))

# Kiểm định chéo theo thời gian để có khoảng dao động
fraud_cv = []
for tr, va in TimeSeriesSplit(n_splits=5).split(Xf_train):
    fold_model = make_classifier().fit(Xf_train[tr], yf_train[tr])
    fraud_cv.append(roc_auc_score(yf_train[va], fold_model.predict_proba(Xf_train[va])[:, 1]))
print(f"ROC-AUC kiểm định chéo: {np.mean(fraud_cv):.4f} ± {np.std(fraud_cv):.4f}")

# Phân vị điểm trên tập train -> ai-service đổi xác suất thành "rủi ro hơn X% giao dịch"
train_scores = classifier.predict_proba(Xf_train)[:, 1]
reference_quantiles = np.quantile(train_scores, np.linspace(0, 1, 1001))

In [ ]:
fraud_metrics = {
    "model_version": MODEL_VERSION,
    "model": "hist_gradient_boosting (có giám sát)",
    "trained_at": now_iso(),
    "risk_output": "phân vị rủi ro so với phân phối điểm tập train (fraud_score_reference.json)",
    "rows": int(len(fraud_df)),
    "known_anomalies": int(fraud_df["isFraud"].sum()),
    "fraud_rate_train": round(fraud_rate, 6),
    "features": FRAUD_FEATURES,
    "dataset": "IEEE-CIS Fraud Detection (Kaggle, Vesta) — giao dịch thẻ thật có nhãn isFraud",
    "split": {
        "strategy": "time_order",
        "train_rows": int(len(fraud_train)),
        "test_rows": int(len(fraud_test)),
    },
    "test_metrics": clf_report,
    "time_series_cv_roc_auc": {"mean": float(np.mean(fraud_cv)), "std": float(np.std(fraud_cv))},
    "rejected_alternatives": {
        "isolation_forest": {
            **iso_report,
            "reason": "Không giám sát, xấp xỉ ngẫu nhiên trên dữ liệu thật: "
            "bất thường thống kê không trùng với gian lận.",
        }
    },
    "ensemble": "risk = max(0.75 * luật + 0.25 * mô hình, luật); "
    "quản trị viên xác nhận mọi cảnh báo",
    "limitations": [
        "failed_payment_count không có trong IEEE-CIS (đặt 0); "
        "tầng luật của hệ thống xử lý tín hiệu này.",
        "Giao dịch thương mại điện tử bằng thẻ, khác hành vi ủng hộ trên nền tảng gây quỹ.",
        "Mô hình chỉ tạo tín hiệu phụ; quản trị viên xác nhận trước mọi hành động.",
    ],
}
joblib.dump(classifier, OUT_DIR / "fraud_classifier.pkl", compress=3)
save_json(
    "fraud_score_reference.json",
    {"model_version": MODEL_VERSION, "quantiles": reference_quantiles.tolist()},
)
save_json("fraud_metrics.json", fraud_metrics)

# Kiểm tra giống ai-service: predict_proba -> phân vị rủi ro
f_model = joblib.load(OUT_DIR / "fraud_classifier.pkl")
for label, row in {"bình thường": [1, 0, 3, 0.2, 300], "đáng ngờ": [25, 0, 40, 6.0, 0]}.items():
    proba = float(f_model.predict_proba(np.asarray([row], dtype=float))[0][1])
    percentile = (
        np.searchsorted(reference_quantiles, proba, side="right") / reference_quantiles.size
    )
    print(f"{label:12s} xác suất={proba:.4f} phân vị rủi ro={percentile:.3f}")

## 4. Gợi ý dự án: học độ tương đồng giữa các lĩnh vực (DonorsChoose)

Thuật toán gợi ý của ai-service cộng điểm cho dự án thuộc lĩnh vực **giống** lĩnh vực người dùng từng quan tâm (`RecIndex.similarity`). Trước đây độ tương đồng này tính từ dữ liệu tự sinh. Ở đây ta học nó từ **hành vi quyên góp thật**: hai lĩnh vực càng hay được cùng một người quyên góp thì càng tương đồng (lọc cộng tác mức lĩnh vực, cosine giữa các cột của ma trận người quyên góp × lĩnh vực).

Bản DonorsChoose này không có ngày quyên góp: mốc thời gian lấy từ **ngày đăng dự án**, thứ tự quyên góp của từng người lấy từ `Donor Cart Sequence` (nếu không có ngày thì giữ lại lần quyên góp cuối của mỗi người làm tập kiểm tra).

Đánh giá ngoại tuyến: với mỗi người quyên góp, dự đoán **lĩnh vực** của lần quyên góp kế tiếp, so sánh 4 cách: ngẫu nhiên, phổ biến toàn cục, chỉ lịch sử cá nhân, lịch sử + độ tương đồng học được (giống công thức ai-service).

> DonorsChoose là dự án lớp học ở Mỹ; lĩnh vực môn học được ánh xạ gần đúng về 6 lĩnh vực của Góp Mầm (ghi trong metrics).

In [ ]:
DONORSCHOOSE_TO_CATEGORY = {
    "Math & Science": "Công nghệ",
    "Applied Learning": "Khởi nghiệp",
    "Literacy & Language": "Giáo dục",
    "History & Civics": "Giáo dục",
    "Special Needs": "Giáo dục",
    "Music & The Arts": "Văn hóa",
    "Health & Sports": "Y tế",
    "Warmth, Care & Hunger": "Y tế",
}


def first_subject(tree: str) -> str:
    # "Warmth, Care & Hunger" có dấu phẩy trong tên -> xét riêng trước khi tách
    text = str(tree)
    if text.startswith("Warmth, Care & Hunger"):
        return "Warmth, Care & Hunger"
    return text.split(",")[0].strip()


def category_similarity(user_category: pd.DataFrame) -> pd.DataFrame:
    # user_category: cột user_id, category
    # -> ma trận cosine giữa các lĩnh vực (0..1, đường chéo = 1)
    matrix = (
        pd.crosstab(user_category["user_id"], user_category["category"]).clip(upper=1).astype(float)
    )
    matrix = matrix.reindex(columns=CATEGORIES, fill_value=0.0)
    vectors = matrix.to_numpy().T
    norms = np.linalg.norm(vectors, axis=1, keepdims=True)
    norms[norms == 0] = 1.0
    unit = vectors / norms
    values = np.clip(unit @ unit.T, 0.0, 1.0)
    np.fill_diagonal(values, 1.0)
    return pd.DataFrame(values, index=CATEGORIES, columns=CATEGORIES).round(4)


def rank_categories(
    history: list[str], method: str, popularity: pd.Series, sim: pd.DataFrame, rng
) -> list[str]:
    if method == "random":
        return list(rng.permutation(CATEGORIES))
    if method == "popular":
        return sorted(CATEGORIES, key=lambda c: popularity.get(c, 0.0), reverse=True)
    interest = {c: 0.0 for c in CATEGORIES}
    for position, category in enumerate(
        reversed(history)
    ):  # mới nhất trước, giảm dần theo thời gian
        interest[category] += 0.9**position
    top = max(interest.values()) or 1.0
    scores = {}
    for c in CATEGORIES:
        score = 0.2 * interest[c] / top + 0.01 * popularity.get(c, 0.0)
        if method == "history_similarity":
            score += 0.35 * max((sim.loc[c, h] for h in set(history) if h != c), default=0.0)
        scores[c] = score
    return sorted(CATEGORIES, key=lambda c: scores[c], reverse=True)

In [ ]:
dc_dir = DATA_DIR / "donorschoose"


def find_csv(stem: str) -> Path:
    # Tìm file không phân biệt hoa/thường (bản sao trên Kaggle có thể đổi tên file)
    matches = [p for p in dc_dir.rglob("*.csv") if p.stem.lower() == stem.lower()]
    if not matches:
        found = sorted(p.name for p in dc_dir.rglob("*.csv"))
        raise FileNotFoundError(f"Không thấy {stem}.csv trong {dc_dir}. Các file có: {found}")
    return matches[0]


def pick_column(path: Path, *candidates: str, required: bool = True) -> str | None:
    # Dò tên cột theo nhiều cách viết (vd. "Project ID" / "project_id" / "projectid")
    columns = pd.read_csv(path, nrows=0).columns
    normalized = {c.lower().replace(" ", "").replace("_", ""): c for c in columns}
    for candidate in candidates:
        key = candidate.lower().replace(" ", "").replace("_", "")
        if key in normalized:
            return normalized[key]
    if required:
        raise KeyError(f"{path.name}: không thấy cột {candidates}. Các cột có: {list(columns)}")
    return None


projects_path, donations_path = find_csv("Projects"), find_csv("Donations")
p_id = pick_column(projects_path, "Project ID", "projectid")
p_cat = pick_column(projects_path, "Project Subject Category Tree")
p_date = pick_column(projects_path, "Project Posted Date", "date_posted", required=False)
d_pid = pick_column(donations_path, "Project ID", "projectid")
d_donor = pick_column(donations_path, "Donor ID", "donorid")
d_seq = pick_column(donations_path, "Donor Cart Sequence", required=False)
d_date = pick_column(donations_path, "Donation Received Date", "donation_timestamp", required=False)
print("Projects:", projects_path.name, [p_id, p_cat, p_date])
print("Donations:", donations_path.name, [d_pid, d_donor, d_seq, d_date])

projects = pd.read_csv(projects_path, usecols=[c for c in (p_id, p_cat, p_date) if c])
projects["category"] = projects[p_cat].map(first_subject).map(DONORSCHOOSE_TO_CATEGORY)
projects = projects.dropna(subset=["category"]).drop_duplicates(subset=[p_id]).set_index(p_id)
project_category = projects["category"]
project_date = pd.to_datetime(projects[p_date], errors="coerce") if p_date else None
del projects

# Đọc Donations theo khúc 1 triệu dòng để không hết RAM
chunks = []
usecols = [c for c in (d_pid, d_donor, d_seq, d_date) if c]
for chunk in pd.read_csv(donations_path, usecols=usecols, chunksize=1_000_000):
    chunk = chunk[chunk[d_pid].isin(project_category.index)]
    if d_date:
        occurred = pd.to_datetime(chunk[d_date], errors="coerce")
    elif project_date is not None:
        occurred = chunk[d_pid].map(project_date)
    else:
        occurred = pd.Series(pd.NaT, index=chunk.index)
    chunks.append(
        pd.DataFrame(
            {
                "user_id": chunk[d_donor].astype(str),
                "category": chunk[d_pid].map(project_category).astype(str),
                "occurred_at": occurred,
                "seq": chunk[d_seq].fillna(0).astype(int) if d_seq else 0,
            }
        )
    )
events = pd.concat(chunks, ignore_index=True)
del chunks
HAS_TIME = bool(events["occurred_at"].notna().mean() > 0.9)
TIME_SOURCE = d_date or (p_date and f"Projects.{p_date}") or "không có"
events = events.sort_values(["occurred_at", "seq"], na_position="last").reset_index(drop=True)
print(f"Lượt quyên góp: {len(events):,} | người quyên góp: {events['user_id'].nunique():,}")
print("Nguồn thời gian:", TIME_SOURCE, "| có mốc thời gian:", HAS_TIME)
print(events["category"].value_counts(normalize=True).round(3))

In [ ]:
if HAS_TIME:
    cut_time = events["occurred_at"].quantile(TRAIN_FRACTION)
    in_test = events["occurred_at"] > cut_time
    split_info = {
        "strategy": "chronological",
        "fraction": TRAIN_FRACTION,
        "cut_time": str(cut_time),
        "time_source": TIME_SOURCE,
    }
else:
    # Không có thời gian: lần quyên góp cuối (theo Donor Cart Sequence) của mỗi người là tập test
    last_seq = events.groupby("user_id")["seq"].transform("max")
    donations_per_user = events.groupby("user_id")["seq"].transform("size")
    in_test = (events["seq"] == last_seq) & (donations_per_user > 1)
    split_info = {"strategy": "leave_last_out", "order": "Donor Cart Sequence"}

order = ["user_id", "occurred_at", "seq"]
rec_train = events[~in_test].sort_values(order, na_position="last")
rec_test = events[in_test].sort_values(order, na_position="last")
similarity = category_similarity(rec_train)
popularity = rec_train["category"].value_counts(normalize=True)
print("Độ tương đồng học được:\n", similarity)

# Người dùng có lịch sử ở train và ít nhất một lượt ở test;
# lấy mẫu tối đa 20.000 người để chạy nhanh
next_category = rec_test.groupby("user_id")["category"].first()
users = pd.Index(rec_train["user_id"].unique()).intersection(next_category.index)
rng = np.random.default_rng(SEED)
if len(users) > 20_000:
    users = pd.Index(rng.choice(users.to_numpy(), size=20_000, replace=False))
history_by_user = (
    rec_train[rec_train["user_id"].isin(users)].groupby("user_id")["category"].apply(list)
)

rec_results = {}
for method in ["random", "popular", "history", "history_similarity"]:
    hits1 = hits2 = mrr = 0.0
    for user in users:
        ranking = rank_categories(history_by_user[user][-20:], method, popularity, similarity, rng)
        rank = ranking.index(next_category[user]) + 1
        hits1 += rank <= 1
        hits2 += rank <= 2
        mrr += 1.0 / rank
    n = len(users)
    rec_results[method] = {"hit@1": hits1 / n, "hit@2": hits2 / n, "mrr": mrr / n}
    print(f"{method:20s} Hit@1={hits1 / n:.4f} Hit@2={hits2 / n:.4f} MRR={mrr / n:.4f}")

In [ ]:
save_json(
    "category_similarity.json",
    {
        "model_version": MODEL_VERSION,
        "trained_at": now_iso(),
        "source": "DonorsChoose.org (Kaggle hanselhansel/donorschoose) — cosine giữa lĩnh vực "
        "theo người quyên góp chung",
        "categories": CATEGORIES,
        "similarity": {a: {b: float(similarity.loc[a, b]) for b in CATEGORIES} for a in CATEGORIES},
    },
)
save_json(
    "recommender_metrics.json",
    {
        "model_version": MODEL_VERSION,
        "generated_at": now_iso(),
        "dataset": {
            "name": "DonorsChoose.org",
            "events": int(len(events)),
            "train_events": int(len(rec_train)),
            "test_events": int(len(rec_test)),
            "users_evaluated": int(len(users)),
            "category_mapping": DONORSCHOOSE_TO_CATEGORY,
        },
        "split": split_info,
        "task": "Dự đoán lĩnh vực của lượt quyên góp kế tiếp (xếp hạng 6 lĩnh vực)",
        "results": rec_results,
        "limitations": [
            "Dự án lớp học ở Mỹ; lĩnh vực ánh xạ gần đúng.",
            "Đánh giá ở mức lĩnh vực, chưa ở mức từng dự án.",
            "Không có ngày quyên góp: thời gian lấy theo ngày đăng dự án.",
        ],
    },
)

## 5. Mô hình riêng cho lĩnh vực Giáo dục (DonorsChoose)

Mô hình ở mục 2 xếp "Giáo dục" chung với nhóm *Xuất bản / Báo chí* của Kickstarter — đó là dự án bán sản phẩm sáng tạo, khác xa kiểu "tủ sách cho điểm trường". DonorsChoose.org là nơi **giáo viên xin tài trợ sách, đồ dùng cho lớp học**: mỗi dự án có số tiền cần, ngày đăng, ngày hết hạn và kết quả **đủ tiền (Fully Funded)** hoặc **hết hạn (Expired)** — rất gần loại dự án giáo dục trên Góp Mầm.

Mục này huấn luyện một mô hình **riêng cho Giáo dục** và so sánh với mô hình Kickstarter trên **cùng tập kiểm tra DonorsChoose** (chia theo thời gian 80/20 như mục 2):

| Đặc trưng | Nguồn DonorsChoose | Góp Mầm tính giống hệt |
|---|---|---|
| `goal_amount` | Project Cost (USD) × 25.000 | Mục tiêu (₫) |
| `duration_days` | Expiration − Posted | Ngày kết thúc − ngày bắt đầu |
| `name_words` | số từ của Project Title | số từ của tên chiến dịch |
| `launch_month`, `launch_weekday` | ngày đăng | ngày bắt đầu (giờ VN) |

Không cần cột lĩnh vực (mọi dự án đều là Giáo dục). Mô hình **chưa được ai-service dùng**: file xuất ra có tiền tố `education_`, chỉ đưa vào hệ thống sau khi so sánh xong.

> Nếu bản DonorsChoose thiếu cột cần thiết, mục này **tự bỏ qua** (in ra danh sách cột để kiểm tra) và các mô hình khác vẫn được tải về bình thường.

In [ ]:
EDU_MODEL_VERSION = "2026.11-edu.1"
# Đặc trưng có ở cả DonorsChoose lẫn Góp Mầm; thứ tự = thứ tự cột của mô hình.
EDU_FEATURES = ["goal_amount", "duration_days", "name_words", "launch_month", "launch_weekday"]
# Góp Mầm cho gây quỹ tối đa 60 ngày — dùng để báo phần dữ liệu nằm trong vùng này.
GOPMAM_MAX_DAYS = 60

edu_cols = {
    "id": pick_column(projects_path, "Project ID", "projectid", required=False),
    "title": pick_column(projects_path, "Project Title", "title", required=False),
    "cost": pick_column(
        projects_path,
        "Project Cost",
        "total_price_excluding_optional_support",
        "total_price_including_optional_support",
        required=False,
    ),
    "posted": pick_column(projects_path, "Project Posted Date", "date_posted", required=False),
    "expires": pick_column(
        projects_path,
        "Project Expiration Date",
        "date_expiration",
        "expiration_date",
        required=False,
    ),
    "status": pick_column(
        projects_path, "Project Current Status", "funding_status", "status", required=False
    ),
}
edu_missing = [key for key in ("cost", "posted", "expires", "status") if edu_cols[key] is None]
EDU_AVAILABLE = not edu_missing
if not EDU_AVAILABLE:
    print(
        f">>> Bỏ qua mô hình Giáo dục: Projects.csv thiếu cột {edu_missing}. "
        f"Các cột có: {list(pd.read_csv(projects_path, nrows=0).columns)}"
    )
else:
    usecols = [c for c in edu_cols.values() if c]
    edu_raw = pd.read_csv(projects_path, usecols=usecols, low_memory=False)
    status = edu_raw[edu_cols["status"]].astype(str).str.strip().str.lower()
    print("Trạng thái dự án:", status.value_counts().head(6).to_dict())
    # Chỉ dự án đã có kết quả: đủ tiền = 1, hết hạn = 0 (bỏ dự án còn đang gây quỹ).
    # "Not funded" chứa chữ "funded" nên xét hết hạn trước.
    expired = status.str.contains("expired|not funded")
    funded = ~expired & (status.str.contains("fully funded|completed") | (status == "funded"))
    edu = pd.DataFrame(
        {
            "posted": pd.to_datetime(edu_raw[edu_cols["posted"]], errors="coerce"),
            "expires": pd.to_datetime(edu_raw[edu_cols["expires"]], errors="coerce"),
            "cost": pd.to_numeric(edu_raw[edu_cols["cost"]], errors="coerce"),
            "title": (
                edu_raw[edu_cols["title"]].fillna("").astype(str)
                if edu_cols["title"]
                else pd.Series("", index=edu_raw.index)
            ),
            "success": np.where(expired, 0, np.where(funded, 1, -1)),
        }
    )
    del edu_raw
    edu = edu[(edu["success"] >= 0) & (edu["cost"] > 0)].dropna(subset=["posted", "expires"])
    edu["goal_amount"] = edu["cost"].astype(float) * USD_TO_VND
    edu["duration_days"] = (
        ((edu["expires"] - edu["posted"]).dt.total_seconds() / 86_400).round().clip(1, 400)
    )
    edu["name_words"] = edu["title"].str.split().str.len().fillna(0).astype(int)
    edu["launch_month"] = edu["posted"].dt.month
    edu["launch_weekday"] = edu["posted"].dt.weekday
    edu = edu.sort_values("posted").reset_index(drop=True)
    edu_train, edu_test = time_split(edu)
    print(
        f"Dự án đã có kết quả: {len(edu):,} | train {len(edu_train):,} "
        f"({edu_train['posted'].min():%Y-%m} → {edu_train['posted'].max():%Y-%m})"
        f" | test {len(edu_test):,} (→ {edu_test['posted'].max():%Y-%m})"
    )
    print(
        "Tỷ lệ đủ tiền train/test:",
        round(edu_train["success"].mean(), 3),
        "/",
        round(edu_test["success"].mean(), 3),
    )
    print("Thời hạn (ngày):", edu["duration_days"].describe().round(1).to_dict())
    print(
        f"Tỷ lệ dự án có thời hạn ≤ {GOPMAM_MAX_DAYS} ngày (vùng Góp Mầm):",
        round(float((edu["duration_days"] <= GOPMAM_MAX_DAYS).mean()), 3),
    )
    print("Mục tiêu (triệu ₫):", (edu["goal_amount"] / 1e6).describe().round(1).to_dict())

In [ ]:
if EDU_AVAILABLE:
    from sklearn.ensemble import HistGradientBoostingClassifier

    Xe_train = edu_train[EDU_FEATURES].to_numpy(dtype=float)
    Xe_test = edu_test[EDU_FEATURES].to_numpy(dtype=float)
    ye_train = edu_train["success"].to_numpy()
    ye_test = edu_test["success"].to_numpy()

    def edu_logistic() -> Pipeline:
        # Mốc tuyến tính: log(mục tiêu, số ngày), số từ, tháng/thứ dạng one-hot (theo EDU_FEATURES).
        pre = ColumnTransformer(
            [
                (
                    "num",
                    Pipeline([("log", FunctionTransformer(np.log1p)), ("scale", StandardScaler())]),
                    [0, 1],
                ),
                ("words", StandardScaler(), [2]),
                ("cat", OneHotEncoder(handle_unknown="ignore"), [3, 4]),
            ]
        )
        return Pipeline([("pre", pre), ("lr", LogisticRegression(max_iter=1000))])

    EDU_DEPLOYABLE = {
        "gradient_boosting": lambda: GradientBoostingClassifier(
            n_estimators=200, max_depth=3, learning_rate=0.1, subsample=0.8, random_state=SEED
        ),
        "hist_gradient_boosting": lambda: HistGradientBoostingClassifier(
            max_iter=300, learning_rate=0.1, random_state=SEED
        ),
    }
    EDU_BASELINES = {
        "majority_class": lambda: DummyClassifier(strategy="prior"),
        "logistic_regression": edu_logistic,
    }

    edu_eval, edu_fitted = {}, {}
    for name, factory in {**EDU_BASELINES, **EDU_DEPLOYABLE}.items():
        model = factory().fit(Xe_train, ye_train)
        edu_fitted[name] = model
        edu_eval[name] = classification_report_dict(ye_test, model.predict_proba(Xe_test)[:, 1])
        print(
            f"{name:24s} ROC-AUC={edu_eval[name]['roc_auc']:.4f} "
            f"PR-AUC={edu_eval[name]['pr_auc']:.4f} Brier={edu_eval[name]['brier']:.4f}"
        )
    edu_best_name = max(EDU_DEPLOYABLE, key=lambda n: edu_eval[n]["roc_auc"])
    edu_best = edu_fitted[edu_best_name]
    print("\nChọn cho Giáo dục:", edu_best_name)

    # --- So sánh chính: mô hình Kickstarter hiện tại chấm CÙNG tập test DonorsChoose ---------
    ks_cols = {
        "category_id": np.full(len(edu_test), CATEGORY_IDS["Giáo dục"], dtype=float),
        **{name: edu_test[name].to_numpy(dtype=float) for name in EDU_FEATURES},
    }
    # Mô hình Kickstarter học thời hạn tối đa ~92 ngày: giới hạn như lúc huấn luyện.
    ks_cols["duration_days"] = np.clip(ks_cols["duration_days"], 1, 92)
    X_ks_on_edu = np.column_stack([ks_cols[name] for name in SUCCESS_FEATURES])
    ks_on_edu = classification_report_dict(ye_test, best_model.predict_proba(X_ks_on_edu)[:, 1])
    print(
        f"\nKickstarter ({best_name}) trên test DonorsChoose: ROC-AUC={ks_on_edu['roc_auc']:.4f} "
        f"Brier={ks_on_edu['brier']:.4f}"
    )
    print(
        f"Mô hình Giáo dục ({edu_best_name}):               ROC-AUC="
        f"{edu_eval[edu_best_name]['roc_auc']:.4f} Brier={edu_eval[edu_best_name]['brier']:.4f}"
    )

    # Riêng các dự án trong vùng Góp Mầm (≤ 60 ngày) — nơi mô hình thực sự được dùng.
    in_range = edu_test["duration_days"].to_numpy() <= GOPMAM_MAX_DAYS
    edu_in_range = None
    if in_range.sum() >= 200 and len(np.unique(ye_test[in_range])) == 2:
        edu_in_range = {
            "rows": int(in_range.sum()),
            "education_model": classification_report_dict(
                ye_test[in_range], edu_best.predict_proba(Xe_test[in_range])[:, 1]
            ),
            "kickstarter_model": classification_report_dict(
                ye_test[in_range], best_model.predict_proba(X_ks_on_edu[in_range])[:, 1]
            ),
        }
        print(
            f"Chỉ dự án ≤ {GOPMAM_MAX_DAYS} ngày ({edu_in_range['rows']:,} dòng): Giáo dục "
            f"ROC-AUC={edu_in_range['education_model']['roc_auc']:.4f} | Kickstarter "
            f"ROC-AUC={edu_in_range['kickstarter_model']['roc_auc']:.4f}"
        )
    else:
        print(f"Quá ít dự án ≤ {GOPMAM_MAX_DAYS} ngày trong tập test để đánh giá riêng.")

    # Kiểm định chéo theo thời gian (như mục 2) trên tối đa 300.000 dự án train gần nhất cho nhanh.
    cv_rows = min(len(Xe_train), 300_000)
    Xe_cv, ye_cv = Xe_train[-cv_rows:], ye_train[-cv_rows:]
    edu_cv = []
    for fold, (tr, va) in enumerate(TimeSeriesSplit(n_splits=5).split(Xe_cv), start=1):
        m = EDU_DEPLOYABLE[edu_best_name]().fit(Xe_cv[tr], ye_cv[tr])
        edu_cv.append(roc_auc_score(ye_cv[va], m.predict_proba(Xe_cv[va])[:, 1]))
        print(f"Fold {fold}: ROC-AUC={edu_cv[-1]:.4f}")
    print(f"ROC-AUC trung bình {np.mean(edu_cv):.4f} ± {np.std(edu_cv):.4f}")

In [ ]:
if EDU_AVAILABLE:
    # Đường hiệu chuẩn của hai mô hình trên cùng tập test DonorsChoose.
    fig, ax = plt.subplots(figsize=(6, 4))
    for label, proba in (
        (f"Giáo dục ({edu_best_name})", edu_best.predict_proba(Xe_test)[:, 1]),
        (f"Kickstarter ({best_name})", best_model.predict_proba(X_ks_on_edu)[:, 1]),
    ):
        frac_pos, mean_pred = calibration_curve(ye_test, proba, n_bins=10)
        ax.plot(mean_pred, frac_pos, "o-", label=label)
    ax.plot([0, 1], [0, 1], "--", color="gray", label="Hiệu chuẩn hoàn hảo")
    ax.set(
        title="Hiệu chuẩn trên test DonorsChoose",
        xlabel="Xác suất dự đoán",
        ylabel="Tỷ lệ đủ tiền thực tế",
    )
    ax.legend()
    plt.tight_layout()
    plt.savefig(OUT_DIR / "education_diagnostics.png", dpi=150)
    plt.show()

    edu_metrics = {
        "model_version": EDU_MODEL_VERSION,
        "best_model": edu_best_name,
        "chosen_by": "roc_auc",
        "trained_at": now_iso(),
        "category": "Giáo dục",
        "dataset": {
            "name": "DonorsChoose.org (Kaggle hanselhansel/donorschoose, Projects.csv)",
            "rows": int(len(edu)),
            "period": f"{edu['posted'].min():%Y-%m-%d} → {edu['posted'].max():%Y-%m-%d}",
            "success_rate": round(float(edu["success"].mean()), 6),
            "label": "Fully Funded = 1, Expired = 0 (bỏ dự án còn đang gây quỹ)",
            "usd_to_vnd": USD_TO_VND,
            "columns": edu_cols,
            "duration_days": edu["duration_days"].describe().round(2).to_dict(),
            "share_within_60_days": round(
                float((edu["duration_days"] <= GOPMAM_MAX_DAYS).mean()), 6
            ),
        },
        "features": EDU_FEATURES,
        "split": {
            "strategy": "time_order",
            "train_rows": int(len(edu_train)),
            "test_rows": int(len(edu_test)),
            "threshold": TRAIN_FRACTION,
        },
        "models": edu_eval,
        "chosen": edu_eval[edu_best_name],
        "kickstarter_model_on_same_test": {"model_version": MODEL_VERSION, **ks_on_edu},
        "within_60_days": edu_in_range,
        "time_series_cv_roc_auc": {
            "rows": int(cv_rows),
            "mean": float(np.mean(edu_cv)),
            "std": float(np.std(edu_cv)),
            "folds": [float(s) for s in edu_cv],
        },
        "limitations": [
            "Dự án lớp học ở Mỹ (giáo viên xin đồ dùng), không phải gây quỹ cộng đồng ở Việt Nam.",
            "Thời hạn DonorsChoose thường dài hơn mức tối đa 60 ngày của Góp Mầm.",
            "Mục tiêu quy đổi theo tỷ giá cố định, chưa điều chỉnh sức mua.",
        ],
    }
    joblib.dump(edu_best, OUT_DIR / "education_success_model.pkl", compress=3)
    save_json("education_success_features.json", EDU_FEATURES)
    save_json("education_success_metrics.json", edu_metrics)
    save_json(
        "education_success_model.metadata.json",
        {
            "model_version": EDU_MODEL_VERSION,
            "feature_schema_version": "edu-1.0.0",
            "trained_at": edu_metrics["trained_at"],
            "features": EDU_FEATURES,
            "best_model": edu_best_name,
            "category": "Giáo dục",
        },
    )

    # Ví dụ: dự án "Tủ sách thiếu nhi cho điểm trường Tả Van" (50 triệu, tên 9 từ, bắt đầu thứ Năm
    # tháng 10) với vài thời hạn — so hai mô hình.
    print("\nTủ sách Tả Van (50 triệu ₫, tên 9 từ, thứ Năm tháng 10):")
    for days in (30, 45, 60):
        row = {"goal_amount": 50_000_000, "duration_days": days, "name_words": 9,
               "launch_month": 10, "launch_weekday": 3}
        p_edu = float(edu_best.predict_proba([[row[f] for f in EDU_FEATURES]])[0][1])
        ks_row = {**row, "category_id": CATEGORY_IDS["Giáo dục"]}
        p_ks = float(best_model.predict_proba([[ks_row[f] for f in SUCCESS_FEATURES]])[0][1])
        print(f"  {days} ngày: mô hình Giáo dục {p_edu:.1%} | mô hình Kickstarter {p_ks:.1%}")

## 6. Tải các file mô hình về máy

Giải nén `gopmam_models.zip` vào `ai-service/models/` của dự án (ghi đè file cùng tên), rồi:

```powershell
docker compose -f infra/compose.yaml restart ai-service
```

Gửi lại các file `success_metrics.json`, `fraud_metrics.json`, `recommender_metrics.json` để cập nhật số liệu trong README.

Các file `education_*` (mô hình riêng cho Giáo dục, mục 5) **chưa** được ai-service đọc: gửi lại `education_success_metrics.json` để so sánh với mô hình Kickstarter trước khi đưa vào hệ thống.

In [ ]:
archive = shutil.make_archive("/content/gopmam_models", "zip", OUT_DIR)
for item in sorted(OUT_DIR.iterdir()):
    print(f"{item.name:32s} {item.stat().st_size / 1024:10.1f} KB")
files.download(archive)